In [1]:
# Add path to param.py. Current working directory assumed to be bip/bip/notebooks

import sys
sys.path.append("./../")
from param import *

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


# Running Example
Let's suppose we are interested in the parameters $\theta \in [0,\infty)$ and $\Sigma \in \mathbb{R}^{2 \times 2}$. The former is a scalar constrained to be nonnegative and the latter is a positive semidefinite (PSD) matrix. We will create a `ParamGroup` object that encodes the parameter set $\{\theta, \Sigma\}$.

# ParamInfo Class
We start by defining two `ParamInfo` objects, one for $\theta$ and one for $\Sigma$. These objects will store the value type, shape, and constraints for each parameter. The value type is the Python type of the individual elements of the parameter.

In [2]:
theta_info = ParamInfo(value_type="float", shape=(), constraint=(0, None))
Sigma_info = ParamInfo(value_type="float", shape=(2,2), constraint="psd")

In [3]:
# Get length of parameter, the number of scalar values that make up the parameter.

theta_len = len(theta_info)
Sigma_len = len(Sigma_info)

print(f"theta length: {theta_len}")
print(f"Sigma length: {Sigma_len}")

theta length: 1
Sigma length: 4


In [4]:
# Print summary of parameters.

print("theta:")
print(theta_info)

print("\nSigma:")
print(Sigma_info)

theta:
value_type: float
shape: ()
constraint: (0, None)
length: 1

Sigma:
value_type: float
shape: (2, 2)
constraint: psd
length: 4


## Other basic properties

In [5]:
# Use NoConstraint for unconstrained parameters.
example_info = ParamInfo(value_type="int", shape=(5,4,2), constraint=NoConstraint)
print(example_info)

value_type: int
shape: (5, 4, 2)
constraint: NoConstraint
length: 40


In [6]:
# Infinite values can alternatively be provided to produce one-sided bounds, but they will be converted to None.
example_info = ParamInfo(value_type="int", shape=(5,4,2), constraint=(-1, np.inf))
print(example_info)

value_type: int
shape: (5, 4, 2)
constraint: (-1, None)
length: 40


In [7]:
# If bounds tuple is provided but does not impose any constraint, will be simplified to NoConstraint.
example_info = ParamInfo(value_type="int", shape=(5,4,2), constraint=(-np.inf, np.inf))
print(example_info)

value_type: int
shape: (5, 4, 2)
constraint: NoConstraint
length: 40


In [8]:
# Simplex constraint is for parameters whose values are in [0,1] and sum to 1.
example_info = ParamInfo(value_type="float", shape=(3,), constraint="simplex")
print(example_info)

value_type: float
shape: (3,)
constraint: simplex
length: 3


In [9]:
# Parameter info can be updated via setters. Validation checks are performed before setting value.
example_info.constraint = NoConstraint
example_info.value_type = "int"
example_info.shape = (6,10,4)

print(example_info)

value_type: int
shape: (6, 10, 4)
constraint: NoConstraint
length: 240


## Argument Checking

### Invalid Types

In [10]:
# Invalid type
theta_info.value_type = "not_a_type"

ValueError: value_type must equal 'float' or 'int', got not_a_type.

### Invalid shape

In [11]:
# Shape is not consistent with constraint.
Sigma_info.shape = (1,2)

ValueError: Constraint 'psd' only valid for square matrix.

### Invalid constraints

In [12]:
# Unrecognized constraint.
Sigma_info.constraint = "not_a_constraint"

ValueError: If constraint is string, must be 'psd' or 'simplex', got not_a_constraint.

In [13]:
# Bound constraint with lower bound larger than upper bound.
Sigma_info.constraint = (1, 0)

ValueError: Invalid bound constraint (1, 0). Lower bound exceeds upper bound.

In [14]:
# Bound constraint of invalid dimension.
Sigma_info.constraint = (0, 1, 4)

TypeError: If constraint is a tuple, must be length 2 containing lower and upper bounds. Got (0, 1, 4).

In [15]:
# For absence of constraint, use NoConstraint, not None. The latter is reserved for missing argument values.
Sigma_info.constraint = None

TypeError: constraint must either be NoConstraint, a string,or a tuple, not <class 'NoneType'>.

In [16]:
# Simplex constraint only allowed when value_type is float.
example_info = ParamInfo(value_type="int", shape=(3,), constraint="simplex")

ValueError: Constraint 'simplex' requires 'float' value_type.

# ParamGroup Class
Let's suppose we are interested in the parameters $\theta \in [0,\infty)$ and $\Sigma \in \mathbb{R}^{2 \times 2}$. The former is a scalar constrained to be nonnegative and the latter is a positive semidefinite (PSD) matrix. We will create a `ParamGroup` object that encodes the parameter set $\{\theta, \Sigma\}$.

In [ ]:
# Set up parameter information for each parameter.
param_info = {"theta": {"type":"float", "size":(), "constraint":(0, None)},
              "Sigma": {"type":"float", "size":(2,2), "constraint": "psd"}}

In [ ]:
# Instantiate ParamGroup object.
param = ParamGroup(param_info)

In [ ]:
# Parameter names (orders alphabetically)
param.get_param_names()

In [ ]:
# Add and remove a new parameter.
new_param_info = {"type":"int", "size":(5,), "constraint": None}

# Add parameter.
param.add_param("new_param_name", new_param_info)
print(param.get_param_names())

# Remove parameter.
param.remove_param("new_param_name")
print(param.get_param_names())

# ParamValue Class
The `ParamValue` class encodes the notion of a specific value a parameter can assume. For the example above, an example of a valid value might be 
$$
\left\{1.0, \begin{bmatrix} 1.0 & 0.0 \\ 0.0 & 1.0 \end{bmatrix} \right\}.
$$

The core of the class is a dictionary with keys corresponding to parameter names and values to the associated parameter values. The class ensures that a given value is of the correct type and size, and satisfies all of the constraints as specified in a `ParamGroup` object. It also features a `to_array` method that converts the parameter value dictionary to an array, with the elements sorted in a canonical order as determined by `ParamGroup`.


In [ ]:
# Instantiate empty ParamValue object.
param_val = ParamValue(param)
print(param_val.value)

In [ ]:
# Set a new value.
val = {"theta": 1.0, "Sigma": np.diag([1,1])}
param_val.value = val
print(param_val.value)

In [ ]:
# Set value upon instantiation.
param_val = ParamValue(param)